# Day 03 미니 프로젝트 · 작업 관리 MCP 서버 만들기
실습에서 익힌 **Tool · Resource · Prompt · in-process Client**를 한 서버에 모아 '작업 관리 MCP 서버'를 완성합니다.
핵심 구현은 **빈칸(TODO)** 입니다 — 실습을 참고해 직접 채우세요. 마지막의 자동 채점으로 기능이 다 동작하는지 점수로 확인합니다.

### 진행
- Part 1 · 서버 뼈대 + 상태 (주어짐)
- Part 2 · Tool 3개 (빈칸) — 실습 Lab 1·5
- Part 3 · Resource (빈칸) — 실습 Lab 3
- Part 4 · Prompt (빈칸) — 실습 Lab 4
- Part 5 · Client 시나리오 (주어짐)
- Part 6 · 자동 채점 — 기능 점수
---

## 0 · 설치

In [ ]:
%pip install -q fastmcp

In [ ]:
import fastmcp
from fastmcp import FastMCP, Client
print("fastmcp", fastmcp.__version__)

## Part 1 · 서버 뼈대 + 상태
서버 객체와 할 일 목록(TASKS)을 준비합니다. 이 위에 도구/리소스/프롬프트를 얹습니다.

In [ ]:
mcp = FastMCP("TaskServer")
TASKS = []   # 각 항목: {"id", "title", "priority", "done"}
print("서버 준비:", mcp.name)

## Part 2 · Tool 3개 (빈칸)
`@mcp.tool` 로 행동을 정의합니다. 타입힌트·docstring이 스키마·설명이 됩니다 (실습 Lab 1·5).

In [ ]:
# TODO(실습 Lab 1·5 참고): @mcp.tool 로 세 도구를 구현하세요.
#   add_task(title, priority="보통") -> dict : TASKS에 {id,title,priority,done:False} 추가 후 반환
#   complete_task(task_id) -> str            : 해당 id의 done=True, 없으면 raise ValueError
#   list_tasks(only_open=False) -> list      : only_open이면 미완료만 반환

# 여기에 구현

print("Tool 등록 완료")

## Part 3 · Resource (빈칸)
`@mcp.resource(uri, description=...)` — 읽기 전용. 상태를 조회만 합니다 (실습 Lab 3).

In [ ]:
# TODO(실습 Lab 3 참고): @mcp.resource("tasks://stats", description=...) 로
#   {"total":..,"done":..,"open":..} 통계를 반환하는 읽기 전용 리소스를 만드세요.

# 여기에 구현

print("Resource 등록 완료")

## Part 4 · Prompt (빈칸)
`@mcp.prompt` — 대화를 세팅하는 재사용 템플릿 (실습 Lab 4).

In [ ]:
# TODO(실습 Lab 4 참고): @mcp.prompt 로 daily_plan(date) 템플릿을 만드세요.
#   미완료 작업 제목들을 모아 "하루 계획을 세워줘" 형태의 문자열을 반환.

# 여기에 구현

print("Prompt 등록 완료")

## Part 5 · Client 시나리오
in-process Client로 서버에 붙어, 등록된 기능을 나열하고 실제로 실행해 봅니다.

In [ ]:
async with Client(mcp) as client:
    print("도구  :", [t.name for t in await client.list_tools()])
    print("리소스:", [str(r.uri) for r in await client.list_resources()])
    print("프롬프트:", [p.name for p in await client.list_prompts()])
    await client.call_tool("add_task", {"title": "슬라이드 검토", "priority": "높음"})
    await client.call_tool("add_task", {"title": "노트북 검증"})
    print("완료 처리:", (await client.call_tool("complete_task", {"task_id": 1})).data)
    print("미완료  :", (await client.call_tool("list_tasks", {"only_open": True})).data)
    s = await client.read_resource("tasks://stats")
    print("통계    :", s[0].text)
    m = await client.get_prompt("daily_plan", {"date": "7/8"})
    print("프롬프트 미리보기:", m.messages[0].content.text[:50], "...")

## Part 6 · 자동 채점 — 기능 점수
빈칸을 제대로 채웠는지 5개 항목으로 채점합니다. 5/5가 목표입니다.

In [ ]:
async def grade():
    score, log = 0, []
    async with Client(mcp) as c:
        tnames = {t.name for t in await c.list_tools()}
        for need in ("add_task", "complete_task", "list_tasks"):
            ok = need in tnames; score += ok; log.append(f"[{'O' if ok else 'X'}] Tool {need}")
        ruris = {str(r.uri) for r in await c.list_resources()}
        ok = "tasks://stats" in ruris; score += ok; log.append(f"[{'O' if ok else 'X'}] Resource tasks://stats")
        pnames = {p.name for p in await c.list_prompts()}
        ok = "daily_plan" in pnames; score += ok; log.append(f"[{'O' if ok else 'X'}] Prompt daily_plan")
    print("\n".join(log)); print(f"\n기능 점수: {score}/5")

await grade()

## Part 7 · 도전 & 산출물 체크리스트
- **A** `set_priority(task_id, priority)` 도구 추가 + 허용값 검증(친절한 에러)
- **B** `tasks://open` 리소스로 미완료만 노출
- **C** `server.py`로 저장 후 `fastmcp dev`로 브라우저 검증

체크리스트
- [ ] Tool 3개(add/complete/list)를 만들어 등록했다
- [ ] 읽기 전용 Resource(tasks://stats)를 만들었다
- [ ] 재사용 Prompt(daily_plan)를 만들었다
- [ ] Client 시나리오가 통과하고 자동 채점 5/5가 나왔다